In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Load error matrices
iontorrent_error = np.load('iontorrent_error_matrix.npy')
illumina_error = np.load('illumina_error_matrix.npy')

nucs = ['A', 'T', 'C', 'G']

In [ ]:
# Load cryptic SNPs
cryptic_snps = pd.read_csv('cryptic_snp_counts.csv', index_col='week')
cryptic_snps.index = pd.to_datetime(cryptic_snps.index)

nuc_to_idx = {'A': 0, 'T': 1, 'C': 2, 'G': 3}


def score_cryptic(snp_type, platform):
    """Score a cryptic SNP by how unexpected it is under the error model.
    Lower error rate -> higher score (more likely to be real). Score in [0, 1]."""
    if platform == 'iontorrent':
        error_matrix = iontorrent_error
    else:
        error_matrix = illumina_error

    ref, alt = snp_type[0], snp_type[2]
    ref_idx = nuc_to_idx[ref]
    alt_idx = nuc_to_idx[alt]
    error_rate = error_matrix[alt_idx, ref_idx]

    # Normalize: mutation with smallest error rate gets score=1
    scalar = error_matrix.min()
    score = (1 / error_rate) * scalar

    return float(score)

for idx, row in cryptic_snps.iterrows():
    if idx <= pd.Timestamp('2024-03-01'):
        platform = 'iontorrent'
    else: 
        platform = 'illumina'
    for col in cryptic_snps.columns:
        cryptic_snps.loc[idx, col] = score_cryptic(col, platform, row[col])

# Set score to the fraction of the total score for that week
cryptic_snps['total_score'] = cryptic_snps.sum(axis=1)

# for col in cryptic_snps.columns:
#     cryptic_snps[col] = cryptic_snps[col].div(cryptic_snps['total_score'], axis=0)

cryptic_snps.to_csv('cryptic_snps_scores.csv')



/tmp/ipykernel_1672470/578087939.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.5533874010353497' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cryptic_snps.loc[idx, col] = score_cryptic(col, platform, row[col])
/tmp/ipykernel_1672470/578087939.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.1559356897461397' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cryptic_snps.loc[idx, col] = score_cryptic(col, platform, row[col])
/tmp/ipykernel_1672470/578087939.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.298296733818091' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  cryptic_snps.loc[idx, col] = sc